# Randomized count calibration

Count-rank calibration fits a reference count distribution on proper-training rows and calibrates its probability ranks on separate observations. This study tests the default score against the original real-width target in [#636](https://github.com/fcela/mboostrs/issues/636), the discrete-set question in [#650](https://github.com/fcela/mboostrs/issues/650), and the zero-inflated gap in [#660](https://github.com/fcela/mboostrs/issues/660). The fitted conditional mean is held fixed across six scores for each of three information tracks.

The [protocol](count_rank_calibration/PROTOCOL.md) fixes the method, five count-generating processes, 23 methods, comparator and promotion gate before implementation or measurement. Each draw has 1,000 analysis observations, a shared 500/500 fitting/calibration partition and 1,000 independent evaluation observations. Ten development draws per process precede 50 fresh primary confirmation draws and a separate 50-draw audit cohort. The [audit addendum](count_rank_calibration/AUDIT_ADDENDUM.md) adds full call capture and requires both confirmation cohorts to pass; it changes no estimator or threshold. Nominal coverage is 0.90, with eligibility window [0.88,0.92]. LightGBM CQR is the prespecified comparator. All cohorts, attempts and unbounded sets are retained.

In [ ]:
from pathlib import Path
import json
import sys

import numpy as np
import polars as pl
from great_tables import GT, md

root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Cargo.toml").exists())
sys.path.insert(0, str(root / "python"))
sys.path.insert(0, str(root))
from benches.evidence_float_codec import decode_record

directory = root / "docs/book/benchmarks/count_rank_calibration"
def read_json(name):
    return decode_record(json.loads((directory / name).read_text()))

primary_summary = read_json("confirmation_summary.json")
primary = primary_summary["rows"]
audit = read_json("confirmation_audit_summary.json")["rows"]
decision = read_json("PROMOTION_DECISION.json")
phases = {"Primary": primary, "Audit": audit}

In [ ]:
def headline_table(rows, cohort):
    leaders = {}
    for panel in {r["panel"] for r in rows}:
        eligible = [r for r in rows if r["panel"] == panel and r.get("coverage_eligible") and not r["failures"]]
        for key in ("width", "count_size", "true_mean_mse", "interval_score"):
            leaders[panel, key] = min((r[key] for r in eligible), default=None)

    def value(row, key):
        if key not in row:
            return "unavailable"
        text = f"{row[key]:.3f}"
        se = row.get(key + "_se")
        if key in ("width", "count_size") and se is not None:
            text += f" ± {se:.3f}"
        if row.get("coverage_eligible") and np.isfinite(row[key]) and row[key] == leaders.get((row["panel"], key)):
            text = f"**{text}**"
        return text

    frame = pl.DataFrame([{ "Regime": r["panel"], "Method": r["method"].replace("_", " "),
        "Width": value(r,"width"), "Count size": value(r,"count_size"), "Coverage": value(r,"coverage"),
        "Mean MSE": value(r,"true_mean_mse"), "Interval score": value(r,"interval_score"),
        "Unbounded": r["unbounded"], "Failed": r["failures"]} for r in rows])
    # Coverage eligibility prevents a narrow but under-covering method from being marked best.
    return (GT(frame, rowname_col="Method", groupname_col="Regime").tab_stubhead("Method")
        .tab_header(title=f"Every method: {cohort} count confirmation",
                    subtitle="50 independent draws per regime; smaller width, cardinality, MSE and score are better")
        .tab_spanner(label="Prediction sets", columns=["Width", "Count size", "Coverage"])
        .fmt_markdown(columns=["Width", "Count size", "Mean MSE", "Interval score"])
        .tab_source_note(md("Width is real length in count units; count size is nonnegative-integer cardinality after "
            "the same support intersection for every method. Width/count show mean ± Monte Carlo SE. Mean MSE "
            "uses the generating conditional mean; CQR points are pre-calibration band midpoints. Score is the 90% "
            "interval score in count units. Bold marks minima among complete coverage-eligible methods. Unbounded "
            "counts valid unbounded draws: their infinite metrics remain infinite, without finite-only averaging. "
            f"Source: {'confirmation' if cohort == 'Primary' else 'confirmation_audit'}_summary.json.")))

In [ ]:
headline_table(primary, "Primary")

## Confirmation targets

Default promotion requires the data-only and supplied fixed original-rate tracks to have eligible coverage and one-sided 95% upper bounds at most 1% for **both real width and count cardinality**. Every default row in every process and all three tracks must also have finite width and eligible coverage. The same gate must pass separately on both confirmation cohorts. The supplied tuned track is an explanatory control and cannot replace either required track.

In [ ]:
targets = []
for cohort, records in phases.items():
    for r in records:
        if r["panel"] == "original" and r["method"].endswith("_auto"):
            targets.append({"Cohort": cohort, "Track": r["method"], "Coverage": r["coverage"],
                "Width excess %": r["width_excess"], "Width upper": r["width_upper95"],
                "Count excess %": r["count_size_excess"], "Count upper": r["count_size_upper95"]})
(GT(pl.DataFrame(targets), rowname_col="Track", groupname_col="Cohort").tab_stubhead("Information track")
 .tab_header(title="Prospective default promotion targets", subtitle="Paired percent excess over LightGBM CQR; smaller is better")
 .tab_spanner(label="Real width", columns=["Width excess %", "Width upper"])
 .tab_spanner(label="Integer cardinality", columns=["Count excess %", "Count upper"])
 .fmt_number(columns=["Coverage", "Width excess %", "Width upper", "Count excess %", "Count upper"], decimals=3)
 .tab_source_note("Upper denotes the one-sided 95% Student-t bound across 50 independent paired draws. "
                  "Each excess is the average of per-draw percent ratios, not a ratio of pooled averages. "
                  "All five competitor comparisons and Bonferroni-adjusted bounds remain in each summary. "
                  "Supplied fixed and data-only tracks are required; supplied tuned is diagnostic."))

In [ ]:
main = next(r for r in primary if r["panel"] == "original" and r["method"] == "discovery_auto")
audited = next(r for r in audit if r["panel"] == "original" and r["method"] == "discovery_auto")
primary_gate = read_json("confirmation_gate.json")
audit_gate = read_json("confirmation_audit_gate.json")
assert primary_gate["cohort_gate_pass"] and audit_gate["cohort_gate_pass"]
assert decision["promote_default"]
print("Both frozen confirmation gates passed; the candidate becomes the automatic count score.")

Data-only default width excess is {eval}`f"{main['width_excess']:.2f}%"` in primary confirmation and {eval}`f"{audited['width_excess']:.2f}%"` in audit confirmation. Their upper bounds are {eval}`f"{main['width_upper95']:.2f}%"` and {eval}`f"{audited['width_upper95']:.2f}%"`. The candidate is promoted to `score="auto"` for eligible standard count models. These results establish the original-rate target in the stated experiment; they do not establish superiority at every rate or for every supplied mean fit.

## Reference distribution and score

For fitted mean $\mu(x)$, the reference is a zero-inflated Poisson with one fitted probability $\pi$ and component mean $\mu(x)/(1-\pi)$. This preserves the mean. The nuisance probability minimizes weighted negative log likelihood using proper-training responses only; it is a score ingredient and does not identify a true zero-generating mechanism. An uninflated Poisson reference is a prespecified diagnostic alternative.

Let $F$ denote the fitted reference cumulative distribution function. An independent auxiliary $U\sim\operatorname{Uniform}[0,1)$ gives jittered outcome $Z=Y+U$ and rank $G=F(Y-1)+U\{F(Y)-F(Y-1)\}$. Randomized ranks for discrete outcomes follow the construction in {cite}`dunn1996randomized`. For each fitted mean, choose a lower-tail probability $a$ on 17 equally spaced points from 0 to $1-\ell$, minimizing the reference quantile span at nominal level $\ell=0.9$. Exact ties choose the smallest $a$.

The calibration score is $S=\max(a-G,G-a-\ell)$. If $q$ is the ordinary split-conformal order statistic of these scores, invert the rank set to obtain $[Q_Z(a-q)-U,Q_Z(a+\ell+q)-U]$ and intersect with nonnegative integer support. Augmented observations $(X,Y,U)$ are exchangeable under the usual split assumptions, so the frozen score yields marginal coverage over both observations and auxiliary randomness even if the reference is wrong. This is the stated discrete-jitter construction; the continuous-outcome distributional conformal paper does not by itself prove discrete conditional coverage. The [protocol](count_rank_calibration/PROTOCOL.md) gives the reference and inversion details.

An upper rank bound at least one yields an infinite endpoint, including under a degenerate fitted reference. An empty count set has ordered singleton real endpoints strictly below zero, and count cardinality zero. This representation is visible in `support`, `count_size` and `empty_count_set`; those rows remain in all metrics. The seed and row order determine the auxiliary draws. Calibration/query streams are separate, and query offsets support reproducible chunking.

The automatic policy uses count ranks when the proper-training residual dispersion ratio $D$ is at most 2, and the existing normalized Poisson score otherwise. Its denominator is $\sum_i w_i\{\mu_i+\pi\mu_i^2/(1-\pi)\}$ and numerator is $\sum_i w_i(Y_i-\mu_i)^2$. This applicability guard is fixed before confirmation, and is neither a variance test nor an attribution of residual noise. Other model families retain their existing routing; explicitly requested legacy scores retain their definitions. CQR and CV+ fitting are unchanged by this feature.

## Public split calibration

The following example repeats a frozen supplied-fixed input. The model and nuisance reference use only fitting rows, while calibration responses determine the cutoff. The returned real interval, count cardinality, selected score and realized seed are public fields. A recorded comparison checks that integration with the latest main branch preserves the frozen predictions.

In [ ]:
import mboostrs as mb
from benches.study_poisson_tuning_consistency import draw
from benches.study_discovery_calibration import partition

analysis, _, _ = draw("original", 6604100)
evaluation, _, _ = draw("original", 6704100)
fitting, calibration, partitions = partition(analysis, 6604100)
model = mb.gamboost("y ~ bols(x1) + bols(x2)", fitting, family=mb.Poisson(),
                    control=mb.boost_control(100, 0.1))
band = mb.conformal_interval(model, evaluation.drop("y"), calibration=calibration,
                             seed=6604100, level=0.9, extrapolation="clip")
example = pl.DataFrame({"Mean": band.fitted_mean[:5], "Lower": band.lower[:5],
                       "Upper": band.upper[:5], "Count size": band.count_size[:5]})
(GT(example).tab_header(title="Automatic count-calibrated predictions", subtitle="Count units; first five query rows")
 .fmt_number(columns=["Mean", "Lower", "Upper"], decimals=3)
 .tab_source_note(f"Resolved score: {band.score}; randomization seed: {band.randomization_seed}. "
                  "This is a repeated recorded input, not a new confirmation draw."))

In [ ]:
# confirmation_summary.json keeps the recorded predictions of this confirmation draw.
recorded = primary_summary["public_workflow_reference"]
assert (recorded["panel"], recorded["seed"], recorded["method"]) == ("original", 6604100, "supplied_fixed_auto")
for key, values in (("lower", band.lower), ("upper", band.upper), ("point", band.fitted_mean)):
    np.testing.assert_allclose(values, recorded["predictions"][key], rtol=0, atol=1e-12)
print("Current public API agrees with all 1,000 recorded supplied-fixed predictions to 1e-12.")

## Audit confirmation

The audit cohort retains every actual native reference-fit, score, cutoff, random-number and endpoint call. Diagnostic observers forward inputs and outputs unchanged. Competitor records retain actual calibration/query predictions, calibration responses, full scores and cutoff. Independent replay compares each distinct native call exactly and reconstructs every competitor interval from these stored predictions. Full coefficients and any coefficient-capture refusal are also retained. The primary cohort remains reported above and is never replaced by the audit.

In [ ]:
headline_table(audit, "Audit")

## Stress cases and limitations

The original-rate gate is distinct from the remaining high-rate and zero-inflated efficiency questions. The deliberately fixed high-rate Poisson mean at 100 updates and rate 0.1 has poor mean predictions; the new score can be wider than a legacy absolute score. Proper mean tuning is therefore consequential. In zero-inflated audit data, discovery improves real width while missing the 1% count-cardinality target. Neither result is removed from the default report, and no score threshold is adjusted after observing it. Low-rate LightGBM CQR is outside the coverage eligibility window, so its narrowness cannot establish eligible comparator leadership.

In [ ]:
stress = []
for cohort, records in phases.items():
    for r in records:
        if r["panel"] in ("high", "zero_inflated") and r["method"].endswith("_auto"):
            stress.append({"Cohort / regime": cohort + " / " + r["panel"], "Method": r["method"],
                "Coverage": r["coverage"], "Width excess %": r["width_excess"], "Width upper": r["width_upper95"],
                "Count excess %": r["count_size_excess"], "Count upper": r["count_size_upper95"]})
(GT(pl.DataFrame(stress), rowname_col="Method", groupname_col="Cohort / regime").tab_stubhead("Information track")
 .tab_header(title="Remaining efficiency gaps", subtitle="Paired percent excess over LightGBM CQR; smaller is better")
 .fmt_number(columns=["Coverage", "Width excess %", "Width upper", "Count excess %", "Count upper"], decimals=3)
 .tab_source_note("Upper is a one-sided 95% bound across draws. These stress rows are reported independently "
                  "of the original-rate promotion target. In particular, #660 remains open for zero-inflated count size."))

## Evidence and reproduction

The corrected development cohort contains 1,150 records; each confirmation cohort contains 5,750 records. Explicit stress score variants can yield valid unbounded sets; infinite width and score remain infinite in aggregation, and their Monte Carlo SE is undefined. All default rows are finite and coverage eligible. The initial development writer rejected nonfinite numbers on three panels; the 498 records written before those transport failures, original logs, failed seeds/methods and original metadata remain archived. A reversible strict-JSON codec distinguishes infinity and NaN from genuine nulls. The broader public export problem is tracked in [#665](https://github.com/fcela/mboostrs/issues/665).

The source manifest verifies runner and loaded-source hashes against the phase-specific frozen commits. The statistical Rust module is identical throughout measurement and final integration. The final native digest differs only because an equivalent R seed-range check was rewritten for strict Clippy. The original measured native identity and the integration identity are both retained. Integration also includes the independently merged CQR changes; this study does not benchmark their performance.

Independent validators check archive hashes, complete method/draw coverage, identical partitions and means across scores, and recompute coverage, real width, integer cardinality, interval score and point errors from every stored prediction, including unbounded outcomes. Native code is checked against SciPy CDF/quantile and likelihood references and both Python/R interfaces. The audit replay below retains its measured and final-integration identities separately.

In [ ]:
development_gate = read_json("development_gate.json")
replay = read_json("SCORE_REPLAY_VALIDATION.json")
validation = pl.DataFrame([
    {"Check": "Development", "Result": str(development_gate["status_counts"])},
    {"Check": "Primary confirmation", "Result": str(primary_gate["status_counts"])},
    {"Check": "Audit confirmation", "Result": str(audit_gate["status_counts"])},
    {"Check": "Distinct native calls replayed exactly", "Result": str(sum(replay["unique_native_calls_replayed"].values()))},
    {"Check": "Competitor intervals reconstructed exactly", "Result": str(sum(replay["controls_reconstructed"].values()))},
    {"Check": "Both-cohort promotion", "Result": str(decision["promote_default"])},
])
(GT(validation).tab_header(title="Complete outcomes and independent checks", subtitle="Unbounded sets remain valid reported outcomes")
 .tab_source_note("Source: phase gate JSON, SCORE_REPLAY_VALIDATION.json and PROMOTION_DECISION.json. "
                  "INTEGRATION_SCORE_REPLAY.json and R parity files retain final-build validation separately."))

Run from a checkout at the phase revision recorded in `SOURCE_MANIFEST.json`, with the matching release extension and repository Python environment. The measurement runner refuses to overwrite evidence; use new output names for a reproduction.

```bash
PYTHONPATH=python:. .venv/bin/python benches/study_count_rank_calibration.py \
  --phase confirmation --panels original --output /tmp/count-original.jsonl.xz
PYTHONPATH=python:. .venv/bin/python benches/summarize_count_rank_calibration.py --phase confirmation
PYTHONPATH=python:. .venv/bin/python benches/verify_count_rank_evidence.py --output /tmp/score-replay.json
MBOOSTRS_R_TEST_LIB=/tmp/private-R-library PYTHONPATH=python:. \
  .venv/bin/python benches/verify_count_rank_r.py
```

The summary command validates the committed complete cohort; repeat with `--phase development` and `--phase confirmation_audit` for the other cohorts. A reproduction on newer code is a separate run. The per-draw archives (assets of the `benchmark-raw-archives-1` GitHub release, listed with their SHA-256 in `docs/book/benchmarks/raw_archives.json`) retain all predictions, partitions, inputs/hashes, warnings, model choices and timing; metadata records platform, package versions, threads and native build. Timing is whole-workflow elapsed cost on shared hardware, not a controlled implementation speed ratio. Memory was not measured. The notebook download contains the protocol, metadata, summary (`development_summary.json`, `confirmation_summary.json`, `confirmation_audit_summary.json`) and validation files.

In [ ]:
artifacts = sorted(p.name for p in directory.iterdir() if p.is_file() and not p.name.endswith((".jsonl.xz", ".jsonl.gz")))
(GT(pl.DataFrame({"Artifact": artifacts}))
 .tab_header(title="Downloaded files", subtitle="Protocols, metadata, summaries and validation")
 .tab_source_note("The per-draw XZ archives (strict JSON with reversible nonfinite locations, decoded with benches.evidence_float_codec) are assets of the benchmark-raw-archives-1 GitHub release, listed in raw_archives.json; the metadata files above describe them."))